## Schritt 1: Daten laden & Netzwerkeigenschaften extrahieren

In [58]:
import numpy as np
import pandas as pd
import networkx as nx
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score

In [59]:
#resting-state Konnektivitätsmatrizen aus der ABIDE-Kohorte (n = 50, Schaefer 100 Parzellierung)

#klinische Informationen in csv-file
df = pd.read_csv('ABIDE_50_subjects_phenotype.csv')

#Konnektivitätsmatrizen in NumPy-Array
conn = np.load('ABIDE_rs_connectivity_50_subjects.npy')

In [60]:
#Format des NumPy-Arrays

print(conn.shape)
#für jede:n Proband:in eine Matrix mit 100 x 100 Einträgen (Korrelationen von 100 Hirnregionen)

#im Array enthaltene Werte sind Korrelationen, daher müssen sie zwischen -1 und 1 liegen
print(conn.min())
print(conn.max())
#Werte zwischen -0.8 und 1

(50, 100, 100)
-0.7994815515147506
1.0


In [61]:
#Extrahieren von 3 graphentheoretischen Netwerk-Metriken (Kennzahlen) pro Proband

n_subjects = conn.shape[0]

#Degree = Anzahl der Verbindungen eines Knotens (Mittelwert pro Proband wird berechnet)
degree_mean = []

#Node Strength = Summe der Verbindungsstärken eines Knotens (Mittelwert pro Proband wird berechnet)
strength_mean = []

#Clustering Coefficient = lokale Vernetzung / wie stark sind die Nachbarn eines Knotens vernetzt (Mittelwert pro Proband wird berechnet)
clustering_mean = []

for i in range(n_subjects):

    # Matrix für einen Probanden
    mat = conn[i]
    mat = np.abs(mat) #Betrag wird genommen (sonst entstehen in der Feature-Matrix komplexe Zahlen)
    mat[mat < 0] = 0  #negative Werte entfernen (sonst entstehen in der Feature-Matrix komplexe Zahlen)
    #Fragen, die unbedingt geklärt werden müssten: 
    #Dürfen die Korrelationen, die kleiner als 0 sind überhaupt entfernt werden? 
    #Wird dadurch die Matrix verfälscht und die Daten unbrauchbar gemacht? 
    #Ist das Klassifikationsmodell noch aussagekräftig?

    # Optional: Diagonale auf 0 setzen
    np.fill_diagonal(mat, 0)

    # Graph erzeugen
    G = nx.from_numpy_array(mat)

    # 1. Degree
    degrees = dict(G.degree())
    degree_mean.append(np.mean(list(degrees.values())))

    # 2. Node Strength (gewichteter Degree)
    strengths = dict(G.degree(weight='weight'))
    strength_mean.append(np.mean(list(strengths.values())))

    # 3. Clustering Coefficient
    clustering = nx.clustering(G, weight='weight')
    clustering_mean.append(np.mean(list(clustering.values())))

# Feature-Matrix (für jeden Proband die drei Mittelwerte)
features = np.column_stack([
    degree_mean,
    strength_mean,
    clustering_mean,
])

In [62]:
print(features.shape)

#Ausgeben der Werte der ersten drei Proband:innen
print(features[:3])
print(features.dtype)
#Komplexe Zahlen wurden durch Schritte weiter oben verhindert
#Frage, die es zu klären gilt: Sind die Werte in der Feature-matrix aussagekräftig?

(50, 3)
[[99.         20.78182188  0.19699009]
 [99.         17.58764148  0.17622997]
 [99.         21.26000541  0.20219882]]
float64


## Schritt 2: Klassifikation mit scikit-learn

In [63]:
#Werte von DX_GROUP (je ein Wert für Autismus-Patient:innen und ein Wert für HCs)
print(df["DX_GROUP"].unique())
#Problem: unklar, welche Gruppe den Wert 1 und welche den Wert 2 zugeordnet hat (keine Information in csv-file)

print(df["DX_GROUP"].value_counts())
#Eine Gruppe enthält 26 Proband:innen, die andere 24.

[1 2]
DX_GROUP
2    26
1    24
Name: count, dtype: int64


In [64]:
#Features und Labels definieren
X = features
y = df["DX_GROUP"].values

#Train- und Test-Datensatz definieren
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

In [65]:
#Pipeline bauen
model = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", RandomForestClassifier(
        n_estimators=200,
        random_state=42
    ))
])

In [66]:
#Model trainieren
model.fit(X_train, y_train)

Pipeline(steps=[('scaler', StandardScaler()),
                ('clf',
                 RandomForestClassifier(n_estimators=200, random_state=42))])

In [67]:
#simple Cross-Validierung 

cv = StratifiedKFold(
    n_splits=5, #5x trainieren und testen
    shuffle=True,
    random_state=42
)

In [68]:
scores = cross_val_score(
    model,
    X,
    y,
    cv=cv,
    scoring="accuracy"
)

In [69]:
print("CV Accuracy Scores:", scores)
#Modellleistung zwischen 40% und 60%

print("Mean Accuracy:", scores.mean())
#Die durchschnittliche Accuracy ist 54% (kaum besser als Zufall).

print("Std Accuracy:", scores.std())

#Fazit: Dieses Modell besitzt nur eine geringe Vorhersagekraft.

CV Accuracy Scores: [0.6 0.6 0.6 0.5 0.4]
Mean Accuracy: 0.5399999999999999
Std Accuracy: 0.07999999999999999
